In [1]:
# Import Modules
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input


In [2]:

# Constants
batch_size = 16
IMG_WIDTH = 256
IMG_HEIGHT = 256
CHANNELS = 3
CLASSES = 21
train_dir = 'UCMerced_LandUse/train_augmented'
validation_dir = 'UCMerced_LandUse/validation'


train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
validation_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

# Load Training Data
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

# Load Validation Data
validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)


Found 10080 images belonging to 21 classes.
Found 840 images belonging to 21 classes.


In [3]:

# Create Model
model_ENB0 = EfficientNetB0(include_top=False, weights='imagenet', input_shape=(IMG_WIDTH, IMG_HEIGHT, CHANNELS))
model_ENB0.trainable = False  # Freeze the pre-trained base model

model = Sequential([
    model_ENB0,
    GlobalAveragePooling2D(),
    Dropout(0.2),
    Dense(CLASSES, activation="softmax")
])

model.summary()

# Compile Model
model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 8, 8, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 21)             │        26,901 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,076,472 (15.55 MB)

 Trainable params: 26,901 (105.08 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [4]:
# Training Parameters
epochs = 10

# Train the Model
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator
)

# Plot Training History 
def plot_training_history(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    plt.figure(figsize=(12, 4))

    # Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(acc, label='Training Accuracy')
    plt.plot(val_acc, label='Validation Accuracy')
    plt.legend()
    plt.title('Accuracy')

    # Loss
    plt.subplot(1, 2, 2)
    plt.plot(loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    plt.legend()
    plt.title('Loss')

    plt.show()

plot_training_history(history)


Epoch 1/10


D:\anaconda3\envs\CS462\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


630/630 ━━━━━━━━━━━━━━━━━━━━ 296s 455ms/step - accuracy: 0.7653 - loss: 0.9875 - val_accuracy: 0.9333 - val_loss: 0.2491
Epoch 2/10
630/630 ━━━━━━━━━━━━━━━━━━━━ 292s 463ms/step - accuracy: 0.9773 - loss: 0.1291 - val_accuracy: 0.9488 - val_loss: 0.1731
Epoch 3/10
630/630 ━━━━━━━━━━━━━━━━━━━━ 287s 455ms/step - accuracy: 0.9884 - loss: 0.0729 - val_accuracy: 0.9607 - val_loss: 0.1511
Epoch 4/10
630/630 ━━━━━━━━━━━━━━━━━━━━ 279s 442ms/step - accuracy: 0.9944 - loss: 0.0494 - val_accuracy: 0.9607 - val_loss: 0.1431
Epoch 5/10
630/630 ━━━━━━━━━━━━━━━━━━━━ 280s 444ms/step - accuracy: 0.9953 - loss: 0.0361 - val_accuracy: 0.9607 - val_loss: 0.1349
Epoch 6/10
 64/630 ━━━━━━━━━━━━━━━━━━━━ 3:53 412ms/step - accuracy: 0.9983 - loss: 0.0291

KeyboardInterrupt: 

Unfreezing from block6d_se_excite :

In [5]:
# Unfreezing
model_ENB0.trainable = True
set_trainable = False

for layer in model_ENB0.layers:
    if layer.name == 'block6d_se_excite':
        set_trainable = True
    if set_trainable:
        if not isinstance(layer, tf.keras.layers.BatchNormalization):  # Correct reference
            layer.trainable = True
        else:
            layer.trainable = False
    else:
        layer.trainable = False

# Updated Model Summary
model.summary()

# Fine-Tuning
model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  # Correct reference
    metrics=['accuracy']
)

# Training Parameters
epochs = 20

# Train the Model
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator,
)

# Plot Accuracy
plt.style.use('fivethirtyeight')
plt.title('Model Accuracy')
plt.plot(range(1, epochs + 1), history.history['accuracy'], label="Training Accuracy")
plt.plot(range(1, epochs + 1), history.history['val_accuracy'], label="Validation Accuracy")
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Plot Loss
plt.style.use('fivethirtyeight')
plt.title('Model Loss')
plt.plot(range(1, epochs + 1), history.history['loss'], label="Training Loss")
plt.plot(range(1, epochs + 1), history.history['val_loss'], label="Validation Loss")
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()




Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 8, 8, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 21)             │        26,901 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,130,276 (15.76 MB)

 Trainable params: 1,369,669 (5.22 MB)

 Non-trainable params: 2,706,803 (10.33 MB)

 Optimizer params: 53,804 (210.18 KB)

Epoch 1/20
630/630 ━━━━━━━━━━━━━━━━━━━━ 314s 483ms/step - accuracy: 0.9930 - loss: 0.0262 - val_accuracy: 0.9667 - val_loss: 0.1235
Epoch 2/20
630/630 ━━━━━━━━━━━━━━━━━━━━ 305s 485ms/step - accuracy: 0.9967 - loss: 0.0116 - val_accuracy: 0.9726 - val_loss: 0.1271
Epoch 3/20
630/630 ━━━━━━━━━━━━━━━━━━━━ 308s 489ms/step - accuracy: 0.9993 - loss: 0.0039 - val_accuracy: 0.9631 - val_loss: 0.1891
Epoch 4/20
630/630 ━━━━━━━━━━━━━━━━━━━━ 315s 499ms/step - accuracy: 0.9968 - loss: 0.0105 - val_accuracy: 0.9714 - val_loss: 0.1393
Epoch 5/20
630/630 ━━━━━━━━━━━━━━━━━━━━ 306s 485ms/step - accuracy: 0.9992 - loss: 0.0032 - val_accuracy: 0.9714 - val_loss: 0.1565
Epoch 6/20
104/630 ━━━━━━━━━━━━━━━━━━━━ 4:00 457ms/step - accuracy: 0.9989 - loss: 0.0027

KeyboardInterrupt: 

In [6]:
# Print all layer names and their indices to inspect naming
for i, layer in enumerate(model_ENB0.layers):
    print(f"Layer {i}: {layer.name}")


Layer 0: input_layer
Layer 1: rescaling
Layer 2: normalization
Layer 3: rescaling_1
Layer 4: stem_conv_pad
Layer 5: stem_conv
Layer 6: stem_bn
Layer 7: stem_activation
Layer 8: block1a_dwconv
Layer 9: block1a_bn
Layer 10: block1a_activation
Layer 11: block1a_se_squeeze
Layer 12: block1a_se_reshape
Layer 13: block1a_se_reduce
Layer 14: block1a_se_expand
Layer 15: block1a_se_excite
Layer 16: block1a_project_conv
Layer 17: block1a_project_bn
Layer 18: block2a_expand_conv
Layer 19: block2a_expand_bn
Layer 20: block2a_expand_activation
Layer 21: block2a_dwconv_pad
Layer 22: block2a_dwconv
Layer 23: block2a_bn
Layer 24: block2a_activation
Layer 25: block2a_se_squeeze
Layer 26: block2a_se_reshape
Layer 27: block2a_se_reduce
Layer 28: block2a_se_expand
Layer 29: block2a_se_excite
Layer 30: block2a_project_conv
Layer 31: block2a_project_bn
Layer 32: block2b_expand_conv
Layer 33: block2b_expand_bn
Layer 34: block2b_expand_activation
Layer 35: block2b_dwconv
Layer 36: block2b_bn
Layer 37: block2b

Unfreezing from block4c_se_excite :

In [ ]:
# Unfreezing
model_ENB0.trainable = True
set_trainable = False

for layer in model_ENB0.layers:
    if layer.name == 'block4c_se_excite':
        set_trainable = True
    if set_trainable:
        if not isinstance(layer, tf.keras.layers.BatchNormalization):  # Correct reference
            layer.trainable = True
        else:
            layer.trainable = False
    else:
        layer.trainable = False

# Updated Model Summary
model.summary()

# Fine-Tuning
model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),  # Correct reference
    metrics=['accuracy']
)

# Training Parameters
epochs = 5

# Train the Model
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator,
)

# Plot Accuracy
plt.style.use('fivethirtyeight')
plt.title('Model Accuracy')
plt.plot(range(1, epochs + 1), history.history['accuracy'], label="Training Accuracy")
plt.plot(range(1, epochs + 1), history.history['val_accuracy'], label="Validation Accuracy")
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Plot Loss
plt.style.use('fivethirtyeight')
plt.title('Model Loss')
plt.plot(range(1, epochs + 1), history.history['loss'], label="Training Loss")
plt.plot(range(1, epochs + 1), history.history['val_loss'], label="Validation Loss")
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 8, 8, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 21)             │        26,901 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,076,472 (15.55 MB)

 Trainable params: 3,730,365 (14.23 MB)

 Non-trainable params: 346,107 (1.32 MB)

Epoch 1/5


D:\anaconda3\envs\CS462\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
